# Task 20 — DocVQA Multimodale: Valutazione

Dataset: `lmms-lab/DocVQA`  
Metriche: **ANLS, VQA accuracy**

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))
import torch, pandas as pd, numpy as np
from datasets import load_dataset
from sklearn.metrics import accuracy_score, f1_score, classification_report
import matplotlib.pyplot as plt

CHECKPOINT_DIR = '../checkpoints/task_20_docvqa/adapter'
BASE_MODEL     = 'unsloth/gemma-3-4bb-it-bnb-4bit'
DATASET_ID     = 'lmms-lab/DocVQA'
MAX_EVAL       = 200
LABEL_MAP      = {}
print('Config caricata ✓')

In [ ]:
ds = load_dataset(DATASET_ID)
test_data = ds.get('test', ds['train'].select(range(MAX_EVAL)))
if len(test_data) > MAX_EVAL: test_data = test_data.select(range(MAX_EVAL))
print(f'Esempi di test: {len(test_data)}')
pd.DataFrame(test_data[:3])

In [ ]:
from unsloth import FastLanguageModel
from peft import PeftModel
base_model, tokenizer = FastLanguageModel.from_pretrained(BASE_MODEL, max_seq_length=512, load_in_4bit=True)
FastLanguageModel.for_inference(base_model)
ft_model, _ = FastLanguageModel.from_pretrained(BASE_MODEL, max_seq_length=512, load_in_4bit=True)
ft_model = PeftModel.from_pretrained(ft_model, CHECKPOINT_DIR)
FastLanguageModel.for_inference(ft_model)
print('Modelli caricati ✓')

In [ ]:
SYSTEM_PROMPT = 'Rispondi alla domanda basandoti sul documento/immagine fornito.'

def predict(model, tokenizer, texts):
    preds = []
    for text in texts:
        prompt = f'<start_of_turn>user\n{SYSTEM_PROMPT}\n\nquestion: {text}\n<end_of_turn>\n<start_of_turn>model\n'
        inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=32, do_sample=False, temperature=1.0)
        decoded = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
        preds.append(decoded.strip())
    return preds

print('Funzione di inferenza pronta ✓')

In [ ]:
texts  = [str(r.get('question', '')) for r in test_data]
y_true = [str(r.get('answers', '')) for r in test_data]

print('Predizioni BASE...')
y_base = predict(base_model, tokenizer, texts)
print('Predizioni FINE-TUNED...')
y_ft   = predict(ft_model,   tokenizer, texts)

# Metriche
try:
    acc_base = accuracy_score(y_true, y_base)
    acc_ft   = accuracy_score(y_true, y_ft)
    f1_base  = f1_score(y_true, y_base, average='macro', zero_division=0)
    f1_ft    = f1_score(y_true, y_ft,   average='macro', zero_division=0)
    print(f'Base  — Acc: {acc_base:.3f}  F1: {f1_base:.3f}')
    print(f'FT    — Acc: {acc_ft:.3f}    F1: {f1_ft:.3f}')
    print(f'Delta — Acc: {acc_ft-acc_base:+.3f}  F1: {f1_ft-f1_base:+.3f}')
except Exception as e:
    print(f'Metriche custom richieste per Task 20: {e}')
    print('Metriche: ANLS, VQA accuracy')

In [ ]:
# Distribuzione predizioni fine-tuned
from collections import Counter
counts = Counter(y_ft)
fig, ax = plt.subplots(figsize=(8,4))
ax.bar(list(counts.keys())[:15], list(counts.values())[:15])
ax.set_title('Task 20 — DocVQA Multimodale: distribuzione predizioni')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.savefig('../checkpoints/task_20_docvqa/evaluation.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
N = 10
df = pd.DataFrame({'Input': [str(t)[:60]+'...' if len(str(t))>60 else str(t) for t in texts[:N]], 'Vera': y_true[:N], 'Base': y_base[:N], 'FT': y_ft[:N]})
df['Base✓'] = df['Vera'] == df['Base']
df['FT✓']   = df['Vera'] == df['FT']
print(df.to_string(index=False))